In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import warnings
from datetime import timedelta

warnings.filterwarnings('ignore')

def load_data():
    print("Loading data...")
    path = '/kaggle/input/competitions/store-sales-time-series-forecasting/'
    train = pd.read_csv(path + 'train.csv', parse_dates=['date'])
    test = pd.read_csv(path + 'test.csv', parse_dates=['date'])
    stores = pd.read_csv(path + 'stores.csv')
    oil = pd.read_csv(path + 'oil.csv', parse_dates=['date'])
    holidays = pd.read_csv(path + 'holidays_events.csv', parse_dates=['date'])
    return train, test, stores, oil, holidays

def prepare_features(train, test, stores, oil, holidays):
    print("Preparing features...")
    
    # Oil price processing
    oil['date'] = pd.to_datetime(oil['date'])
    oil = oil.set_index('date').resample('D').mean().interpolate(method='linear').reset_index()
    oil['oil_lags_1'] = oil['dcoilwtico'].shift(1)
    oil['oil_rolling_7'] = oil['dcoilwtico'].rolling(7).mean()
    oil['oil_rolling_14'] = oil['dcoilwtico'].rolling(14).mean()
    
    # Holidays processing
    holidays = holidays[holidays['transferred'] == False]
    holidays = holidays.drop_duplicates(subset=['date'])
    
    # Combine train and test
    df = pd.concat([train, test], axis=0)
    df['date'] = pd.to_datetime(df['date'])
    
    # Merge datasets
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    df = df.merge(holidays[['date', 'type', 'locale', 'locale_name']], on='date', how='left')
    
    # Date features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_payday'] = df['day_of_month'].isin([15, 30, 31]).astype(int)
    
    # Categorical encoding
    le = LabelEncoder()
    cat_cols = ['family', 'city', 'state', 'type_x', 'type_y', 'locale', 'locale_name']
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
        
    # Target transformation
    df['sales'] = np.log1p(df['sales'])
    
    # Advanced Lags: Only use lags >= 16 to avoid data leakage in test set (16 days)
    print("Creating lag features...")
    for lag in [16, 17, 18, 19, 20, 21, 22, 28, 30, 35, 42, 60, 90, 365]:
        df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)
        
    # Rolling averages (shifted by 16 to be safe)
    for window in [7, 14, 28]:
        df[f'rolling_mean_{window}'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
            lambda x: x.shift(16).rolling(window).mean()
        )
        
    return df

def train_and_predict(df):
    print("Training models...")
    # Use data from 2017 onwards for training to capture recent trends
    train_df = df[(df['date'] >= '2017-01-01') & (df['date'] < '2017-08-16')].dropna()
    test_df = df[df['date'] >= '2017-08-16']
    
    features = [col for col in df.columns if col not in ['id', 'date', 'sales', 'dcoilwtico']]
    
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': 0.02,
        'num_leaves': 128,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.7,
        'bagging_freq': 5,
        'max_depth': 10,
        'min_data_in_leaf': 20,
        'seed': 42
    }
    
    # Multi-seed ensemble for stability and performance
    seeds = [42, 123, 777]
    test_preds = np.zeros(len(test_df))
    
    for seed in seeds:
        print(f"Training with seed {seed}...")
        params['seed'] = seed
        model = lgb.LGBMRegressor(**params, n_estimators=2000)
        model.fit(
            train_df[features], train_df['sales'],
            eval_set=[(train_df[features], train_df['sales'])],
            callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=200)]
        )
        test_preds += np.expm1(model.predict(test_df[features])) / len(seeds)
        
    return test_df['id'], test_preds

def main():
    train, test, stores, oil, holidays = load_data()
    df = prepare_features(train, test, stores, oil, holidays)
    ids, preds = train_and_predict(df)
    
    submission = pd.DataFrame({'id': ids, 'sales': preds})
    submission['sales'] = submission['sales'].clip(0, None)
    submission.to_csv('submission.csv', index=False)
    print("Success! Submission saved to submission.csv")

if __name__ == "__main__":
    main()


Loading data...
Preparing features...
Creating lag features...
Training models...
Training with seed 42...
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.414174
[400]	valid_0's rmse: 0.389822
[600]	valid_0's rmse: 0.380039
[800]	valid_0's rmse: 0.372492
[1000]	valid_0's rmse: 0.3661
[1200]	valid_0's rmse: 0.360311
[1400]	valid_0's rmse: 0.354898
[1600]	valid_0's rmse: 0.349785
[1800]	valid_0's rmse: 0.344913
[2000]	valid_0's rmse: 0.340278
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 0.340278
Training with seed 123...
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.413636
[400]	valid_0's rmse: 0.389818
[600]	valid_0's rmse: 0.379754
[800]	valid_0's rmse: 0.372436
[1000]	valid_0's rmse: 0.36587
[1200]	valid_0's rmse: 0.360204
[1400]	valid_0's rmse: 0.354803
[1600]	valid_0's rmse: 0.349403
[1800]	valid_0's rmse: 0.344608
[2000]	valid_0's rmse: 0.339768
Did not meet early stopping. Best 